In [1]:
!kaggle datasets download -d solarmainframe/ids-intrusion-csv
!unzip ids-intrusion-csv.zip

Dataset URL: https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv
License(s): Attribution 4.0 International (CC BY 4.0)
 99% 1.59G/1.60G [00:19<00:00, 133MB/s]
100% 1.60G/1.60G [00:19<00:00, 87.8MB/s]
Archive:  ids-intrusion-csv.zip
  inflating: 02-14-2018.csv          
  inflating: 02-15-2018.csv          
  inflating: 02-16-2018.csv          
  inflating: 02-20-2018.csv          
  inflating: 02-21-2018.csv          
  inflating: 02-22-2018.csv          
  inflating: 02-23-2018.csv          
  inflating: 02-28-2018.csv          
  inflating: 03-01-2018.csv          
  inflating: 03-02-2018.csv          


In [ ]:
!pip install transformers datasets torch
!pip install trl

In [2]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import scapy.all as scapy

# Load tokenizer and pre-trained model
MODEL_NAME = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

# Define TANTGen model based on GPT-2
class TANTGen(nn.Module):
    def __init__(self, base_model):
        super(TANTGen, self).__init__()
        self.transformer = base_model

    def forward(self, input_ids, attention_mask=None):
        return self.transformer(input_ids, attention_mask=attention_mask).logits

# Create model instance
tantgen = TANTGen(model)

def generate_adversarial_traffic(prompt, max_length=50):
    """ Gera tráfego adversarial a partir de um prompt de entrada. """
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    output = tantgen.transformer.generate(input_ids, max_length=max_length)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example usage
prompt = "TCP SYN packet for firewall bypass: "
adv_traffic = generate_adversarial_traffic(prompt)
print("Generated Adversarial Traffic:", adv_traffic)

# Define training hyperparameters
optimizer = optim.AdamW(tantgen.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

def train_tantgen(data, epochs=3):
    """ Simple function to train the model with a dataset."""
    tantgen.train()
    for epoch in range(epochs):
        for sample in data:
            inputs = tokenizer(sample, return_tensors="pt", padding=True, truncation=True)
            outputs = tantgen(**inputs)
            loss = criterion(outputs.view(-1, outputs.size(-1)), inputs["input_ids"].view(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Example dataset of traffic
training_data = [
    "Modified ICMP packet to evade IDS.",
    "Injection of TCP packets to simulate legitimate traffic.",
    "DDoS attack with fragmented UDP packets."
]

# Train the model
train_tantgen(training_data, epochs=3)

# Function to manipulate network packets using Scapy
def modify_packet(packet):
    """ Modifies network packets to create adversarial examples. """
    if packet.haslayer(scapy.IP):
        packet[scapy.IP].ttl = 64  # Example modification
    return packet

# Example of capturing and modifying packets
def capture_and_modify():
    """ Captures network packets and applies modifications. """
    packets = scapy.sniff(count=10)
    modified_packets = [modify_packet(pkt) for pkt in packets]
    scapy.wrpcap("modified_traffic.pcap", modified_packets)

capture_and_modify()


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"  # You can choose other GPT models like 'gpt2-medium', 'gpt2-large', or 'gpt2-xl'
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

from datasets import Dataset
from transformers import Trainer, TrainingArguments

# Example data (input-output pairs)
data = [
    {"input": "What is the capital of France?", "output": "The capital of France is Paris."},
    # Add more examples
]

# Tokenizing the input-output pairs
def tokenize_function(examples):
    inputs = tokenizer(examples['input'], truncation=True, padding="max_length", max_length=64)
    outputs = tokenizer(examples['output'], truncation=True, padding="max_length", max_length=64)
    inputs['labels'] = outputs['input_ids']
    return inputs

dataset = Dataset.from_dict(data)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./gpt_finetuned_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=500,
    warmup_steps=100,
)

# Set up Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Fine-tune the model
trainer.train()

model.save_pretrained("./fine_tuned_gpt")
tokenizer.save_pretrained("./fine_tuned_gpt")

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"  # Or use any pre-trained GPT model from Hugging Face
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

from datasets import load_dataset

# Example: Load a large text dataset (like Wikipedia or any custom corpus)
dataset = load_dataset("wikipedia", "20200501.en", split='train[:10%]')  # Using a small subset for demonstration

# Tokenize the text data
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

from transformers import Trainer, TrainingArguments

# Set up training arguments for pre-training
training_args = TrainingArguments(
    output_dir="./gpt_pretrained_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=500,
    warmup_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Train the model (this can take a long time depending on the size of your dataset and hardware)
trainer.train()

model.save_pretrained("./gpt_pretrained_model")
tokenizer.save_pretrained("./gpt_pretrained_model")

model = GPT2LMHeadModel.from_pretrained("./gpt_pretrained_model")
tokenizer = GPT2Tokenizer.from_pretrained("./gpt_pretrained_model")
# train

In [ ]:
from transformers import AutoModelForCausalLM
from trl import PPOTrainer, PPOConfig

model = AutoModelForCausalLM.from_pretrained("gpt2")

ppo_config = PPOConfig(
    model_name="gpt2",
    learning_rate=1e-5,
    batch_size=16,
)

from transformers import pipeline

sentiment_pipe = pipeline("sentiment-analysis")

def reward_function(outputs):
    # Example: Reward high positive sentiment
    sentiments = sentiment_pipe(outputs)
    return [1.0 if s["label"] == "POSITIVE" else 0.0 for s in sentiments]

ppo_trainer = PPOTrainer(
    model=model,
    config=ppo_config,
    tokenizer=tokenizer,
    reward_function=reward_function,
)

texts = ["Sample input for the model..."]

for epoch in range(num_epochs):
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    query_tensors = batch["input_ids"]
    response_tensors = model.generate(query_tensors)
    rewards = reward_function(tokenizer.batch_decode(response_tensors, skip_special_tokens=True))
    ppo_trainer.step(query_tensors, response_tensors, rewards)